# Data Manipulation — Basic
## Data Cleaning & Quality Control

**Philosophy:** This note is pattern-first, not syntax-first. It assumes you know the Pandas/NumPy API. Each section answers: *you have data that looks like X — here is the full pipeline to fix it.*

---

## Decision Table

| If your data has... | Go to |
| :--- | :--- |
| Unknown shape, types, or quality | §1 — Audit |
| Dates as strings, IDs as floats, booleans as objects | §2 — Fix dtypes |
| Missing values and unsure what to do | §3 — Missing Values |
| Suspected duplicate rows | §4 — Duplicates |
| Inconsistent category labels (`"US"`, `"us"`, `" USA "`) | §5 — String & Categorical |
| Extreme values distorting results | §6 — Outliers |
| Column names with spaces, caps, or special characters | §7 — Column Names |

## Series Map

| Notebook | Use it for |
| :--- | :--- |
| `DM_Basic` **(this note)** | Cleaning & QC — audit, dtypes, nulls, dupes, strings, outliers, column names |
| `DM_Intermediate` | Reshaping & combining — wide/long, groupby, multi-table joins, time-series, nested, many files |
| `DM_Advanced` | Feature engineering — encoding, transforms, binning, date, lag/window, interactions, text, leakage |
| `Reference_Pandas / Reference_NumPy` | Syntax lookups for individual methods |


> **Environment:** pandas 3.x / numpy 2.x. Under pandas 3.0 **Copy-on-Write** is the default, so chained assignment (`df[mask]['col'] = v`) is a silent no-op — use `df.loc[mask, 'col'] = v`. `inplace=True` no longer saves time or memory; prefer reassignment. A sample frame is defined below so the patterns are runnable.

In [1]:
import pandas as pd
import numpy as np

# Sample messy frame to run the cleaning patterns against.
# Replace with your own:  df = pd.read_csv('data.csv')
df = pd.DataFrame({
    'user_id':    [1, 2, 3, 4, 4, 5],
    'event_date': ['2024-01-05','2024-01-06','2024/01/07','08-01-2024','08-01-2024', None],
    'age':        [34, 28, -5, 41, 41, 130],
    'revenue':    ['$1,200', '950', '', '2,000', '2,000', '15'],
    'country':    ['US', ' us ', 'USA', 'Canada', 'Canada', 'cnaada'],
    'platform':   ['iOS','Android','Web','iOS','iOS','Web'],
})
print(df)


   user_id  event_date  age revenue country platform
0        1  2024-01-05   34  $1,200      US      iOS
1        2  2024-01-06   28     950     us   Android
2        3  2024/01/07   -5             USA      Web
3        4  08-01-2024   41   2,000  Canada      iOS
4        4  08-01-2024   41   2,000  Canada      iOS
5        5         NaN  130      15  cnaada      Web


---
## §1 — Audit a New Dataset

Run this sequence every time you load a new dataset — before touching anything. The goal is to build a complete picture of what you have and what needs fixing.

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('data.csv')

# ── Step 1: Shape and preview ───────────────────────────────────────────────
print(df.shape)          # How many rows and columns?
df.head()                # Does the data look as expected?
df.tail()                # Does the end of the file look clean (no trailing junk rows)?

# ── Step 2: Types and nulls ─────────────────────────────────────────────────
df.info()                # Dtypes + non-null count — the fastest signal of type problems

# ── Step 3: Null summary ────────────────────────────────────────────────────
null_summary = pd.DataFrame({
    'null_count': df.isna().sum(),
    'null_rate':  (df.isna().sum() / len(df)).round(3)
}).query('null_count > 0').sort_values('null_rate', ascending=False)
print(null_summary)
# Interpretation:
# null_rate < 0.05  → likely safe to drop or fill
# null_rate > 0.50  → question whether the column is usable at all

# ── Step 4: Duplicate check ─────────────────────────────────────────────────
print('Full duplicates:',    df.duplicated().sum())
print('Key-level duplicates:', df.duplicated(subset=['user_id', 'date']).sum())

# ── Step 5: Cardinality and distributions ───────────────────────────────────
df.describe()                              # Numeric: spot impossible min/max
df.describe(include='object')              # Categorical: top value and frequency

for col in df.select_dtypes('object').columns:
    print(f"{col}: {df[col].nunique()} unique — {df[col].value_counts().head(3).to_dict()}")
# Look for: unexpectedly high cardinality (ID masquerading as category)
# and low cardinality numerics that should be categorical

# ── Step 6: Value range sanity checks ──────────────────────────────────────
print(df['age'].min(), df['age'].max())    # Negative age? Age > 120?
print(df['revenue'].min())                 # Negative revenue — refund or error?
print((df['rate'] > 1).sum())             # Rate column — should be 0–1 or 0–100?

**What each step is looking for:**

| Step | Red flags to catch |
| :--- | :--- |
| Shape + preview | Extra header rows, merged cells exported from Excel, trailing commas |
| `.info()` | Date columns as `object`, ID columns as `float64` (signals nulls present), boolean as `int` |
| Null summary | Columns with >50% nulls — question if they're usable |
| Duplicates | Full duplicates = data pipeline bug; key-level = fan-out from a join |
| Cardinality | String columns with thousands of unique values — probably IDs, not categories |
| Value ranges | Impossible values: negative age, rate > 100%, future dates in historical data |

**Common mistakes:**
- Skipping the audit and jumping straight to analysis — silent errors propagate and are hard to trace back
- Using `df.describe()` alone — it silently skips non-numeric columns; always pair with `df.info()` and a categorical loop
- Treating a high null rate as always bad — sometimes nulls are meaningful (e.g. `cancellation_date` is null = not cancelled)

**Three more audit checks for §1:** detect columns that secretly mix Python types, measure true memory use, and turn ad-hoc checks into reusable assertions.

In [2]:
import pandas as pd, numpy as np
s = pd.DataFrame({'id':[1,2,3,4],
                  'mixed':[1, 'two', 3, 4.0],            # object col holding int/str/float
                  'platform':['ios','android','ios','web'],
                  'revenue':[10.0, 20.0, np.nan, 40.0]})

# 1) Columns that secretly mix Python types — a silent source of sort/compare bugs
def mixed_type_cols(df):
    out={}
    for c in df.select_dtypes('object').columns:
        kinds = df[c].dropna().map(lambda v: type(v).__name__).unique().tolist()
        if len(kinds) > 1: out[c]=kinds
    return out
print('mixed-type columns:', mixed_type_cols(s))

# 2) True memory footprint — deep=True counts actual string bytes, not just pointers
print(s.memory_usage(deep=True).to_dict())

# 3) Reusable validation: promote ad-hoc checks to assertions you re-run after each step
def validate(df):
    assert df['id'].is_unique,                  'id must be unique'
    assert df['revenue'].dropna().ge(0).all(),  'revenue must be non-negative'
    assert set(df['platform']) <= {'ios','android','web'}, 'unexpected platform'
    print('all checks passed')
validate(s)


mixed-type columns: {'mixed': ['int', 'str', 'float']}
{'Index': 132, 'id': 32, 'mixed': 156, 'platform': 212, 'revenue': 32}
all checks passed


---
## §2 — Fix dtypes

Wrong dtypes are the silent cause of failed joins, incorrect aggregations, and broken date math. Fix them immediately after the audit — everything downstream depends on correct types.

In [ ]:
# ── Pattern 1: Date columns loaded as strings ────────────────────────────────
# Symptom: df.info() shows dtype=object for a date column
df['event_date'] = pd.to_datetime(df['event_date'])                    # auto-detect format
df['event_date'] = pd.to_datetime(df['event_date'], format='%Y-%m-%d') # explicit — faster + safer
df['ts']         = pd.to_datetime(df['ts'], unit='ms')                 # Unix ms timestamp

# Verify:
assert df['event_date'].dtype == 'datetime64[ns]', "Date parse failed"

# ── Pattern 2: ID columns loaded as float (signals nulls were present) ───────
# Symptom: user_id shows as float64 in .info()
# Why: pandas upcasts int → float when nulls exist (int can't hold NaN natively)
df['user_id'] = df['user_id'].fillna(-1).astype(int)   # fill nulls first, then cast
df['user_id'] = df['user_id'].astype(str)              # or keep as string if used as a key

# ── Pattern 3: Boolean columns loaded as int or object ──────────────────────
df['is_active'] = df['is_active'].astype(bool)         # from 0/1 int
df['is_active'] = df['is_active'].map({'True': True, 'False': False, 'true': True, 'false': False})

# ── Pattern 4: Numeric columns loaded as object (has non-numeric chars) ─────
# Symptom: revenue shows as object — likely has '$', ',', or '' values
df['revenue'] = (
    df['revenue']
    .str.replace(r'[\$,]', '', regex=True)   # strip currency symbols and commas
    .str.strip()
    .replace('', np.nan)                     # empty string → NaN before casting
    .astype(float)
)

# ── Pattern 5: Low-cardinality strings as object → category ─────────────────
# When: column has < ~50 unique values and appears in groupby/filter frequently
for col in ['platform', 'region', 'status']:
    if df[col].nunique() < 50:
        df[col] = df[col].astype('category')
# Memory saving: 'category' dtype stores integers internally with a lookup table

# ── Pattern 6: Cast multiple columns at once ────────────────────────────────
type_map = {
    'age':      int,
    'score':    float,
    'user_id':  str,
    'platform': 'category'
}
df = df.astype({k: v for k, v in type_map.items() if k in df.columns})

**dtype fix decision tree:**

```
Column shows as object in .info()?
├── Looks like a date                → pd.to_datetime()
├── Has $, %, commas in values       → str.replace() then .astype(float)
├── Has True/False/Yes/No strings    → .map() then .astype(bool)
├── Few unique values (<50)          → .astype('category')
└── Should be numeric but has blanks → .replace('', np.nan) then .astype(float)

Numeric column shows as float but should be int?
└── Fill or drop NaNs first, then .astype(int)
```

**Common mistakes:**
- `df['col'].astype(int)` when the column has NaNs — raises `ValueError`; fill or drop nulls first
- Parsing ambiguous date formats like `01/02/03` without specifying `format=` — pandas guesses and often guesses wrong
- Converting ID columns to `int` when they have leading zeros (zip codes, product codes) — use `str` instead

**Timezone handling (a §2 dtype landmine).** A naive timestamp carries no zone; mixing tz-aware and tz-naive values in a comparison or join raises. `tz_localize` *attaches* a zone (use once, on naive data); `tz_convert` *changes* an already-aware one.

In [3]:
import pandas as pd
naive = pd.to_datetime(pd.Series(['2024-06-10 14:30', '2024-06-10 18:30']))
print('naive dtype     :', naive.dtype)
aware = naive.dt.tz_localize('America/New_York')   # attach a zone (naive -> aware)
print('tz-aware dtype  :', aware.dtype)
print('converted to UTC:', aware.dt.tz_convert('UTC').tolist())


naive dtype     : datetime64[us]
tz-aware dtype  : datetime64[us, America/New_York]
converted to UTC: [Timestamp('2024-06-10 18:30:00+0000', tz='UTC'), Timestamp('2024-06-10 22:30:00+0000', tz='UTC')]


---
## §3 — Handle Missing Values

The syntax for filling/dropping nulls is in the Pandas reference (§3). This section focuses on the decision: **which strategy to use and when.**

In [ ]:
# ── Strategy 1: Drop — when nulls are random and few ────────────────────────
# Use when: null_rate < 5% and nulls appear to be random (not systematic)
# Risk: if nulls are NOT random, dropping introduces bias
df = df.dropna(subset=['user_id', 'event_date'])   # only drop on key columns

# ── Strategy 2: Fill with a constant ────────────────────────────────────────
# Use when: null means "zero" or "unknown" by business logic
df['revenue'].fillna(0)            # null revenue → 0 spend (not missing, literally zero)
df['category'].fillna('Unknown')   # null category → explicit Unknown label

# ── Strategy 3: Fill with a statistic ───────────────────────────────────────
# Use when: value is missing at random, distribution is roughly symmetric
df['age'].fillna(df['age'].median())              # median — robust to skew
df['score'].fillna(df['score'].mean())            # mean — only if no outliers
df['platform'].fillna(df['platform'].mode()[0])   # mode — for categoricals

# ── Strategy 4: Group-conditional fill ──────────────────────────────────────
# Use when: the right fill value depends on a grouping variable
# Example: fill missing salary with the median salary for that job title
df['salary'] = df.groupby('job_title')['salary'].transform(
    lambda x: x.fillna(x.median())
)

# ── Strategy 5: Forward / backward fill ─────────────────────────────────────
# Use when: data is time-ordered and nulls should carry the last known value
df = df.sort_values(['user_id', 'date'])
df['subscription_status'] = df.groupby('user_id')['subscription_status'].ffill()

# ── Strategy 6: Flag then fill ──────────────────────────────────────────────
# Use when: the fact that a value is missing is itself informative
# (common in churn models — missing last_login_date may signal disengagement)
df['income_missing'] = df['income'].isna().astype(int)   # 1 = was missing
df['income'] = df['income'].fillna(df['income'].median()) # then fill for modeling

# ── Strategy 7: Drop the column entirely ────────────────────────────────────
# Use when: null_rate > 60–70% and no business justification to keep it
threshold = 0.6
cols_to_drop = df.columns[df.isna().mean() > threshold].tolist()
df = df.drop(columns=cols_to_drop)
print(f'Dropped {len(cols_to_drop)} columns: {cols_to_drop}')

**Strategy selection guide:**

| Null rate | Null mechanism | Recommended strategy |
| :--- | :--- | :--- |
| < 5% | Random | Drop the rows |
| Any | Business-defined zero | Fill with 0 or 'Unknown' |
| Any | Time-ordered carry-forward | Forward fill (sort first) |
| Any | Depends on group | Group-conditional fill |
| Any | Missingness is informative | Flag + fill |
| > 60% | Any | Drop the column |

**Common mistakes:**
- Filling with the global mean when group means differ significantly — always check if group-conditional fill is more appropriate
- Forward-filling without sorting by date first — fills in row order, not time order, producing wrong values
- Dropping rows with any null (`dropna()`) when only a few columns matter — use `subset=` to be precise
- Forgetting to add a missingness flag before filling when the null pattern is predictive

---
## §4 — Handle Duplicates

Two distinct types of duplicates — full row duplicates (data pipeline bugs) and key-level duplicates (fan-out from joins or intentional multi-row data). They require different fixes.

In [ ]:
# ── Detect duplicates ────────────────────────────────────────────────────────
# Full duplicates — every column is identical
n_full = df.duplicated().sum()
print(f'Full duplicates: {n_full} ({n_full/len(df):.1%})')

# Key-level duplicates — same key but potentially different other columns
key = ['user_id', 'order_id']
n_key = df.duplicated(subset=key).sum()
print(f'Key-level duplicates: {n_key}')

# Inspect the duplicated records before acting
dup_mask = df.duplicated(subset=key, keep=False)   # keep=False flags ALL copies
df[dup_mask].sort_values(key).head(20)             # look at them side by side

# ── Fix 1: Drop full duplicates ──────────────────────────────────────────────
# Use when: rows are identical — clearly a pipeline or loading bug
df = df.drop_duplicates()

# ── Fix 2: Keep first or last occurrence ─────────────────────────────────────
# Use when: one record per key is correct, duplicates are re-submissions or updates
df = df.sort_values('updated_at', ascending=False)   # latest record first
df = df.drop_duplicates(subset=['user_id'], keep='first')  # keep the most recent

# ── Fix 3: Aggregate to resolve ─────────────────────────────────────────────
# Use when: duplicates exist because one user has multiple rows that should be summed
df = (
    df.groupby('user_id', as_index=False)
      .agg(
          total_spend  = ('spend',   'sum'),
          visit_count  = ('visit_id','count'),
          last_seen    = ('date',    'max')
      )
)

# ── Fix 4: Diagnose why duplicates exist ────────────────────────────────────
# Check if duplication is caused by a join fanout
# Before join:
print('Users before join:', df_users['user_id'].nunique())
merged = df_users.merge(df_orders, on='user_id', how='left')
print('Users after join: ', merged['user_id'].nunique())
# If these differ, the right table has multiple rows per user_id
# Check the right table:
print(df_orders.groupby('user_id').size().describe())

**Duplicate type decision tree:**

```
Are all columns identical?
├── Yes → data pipeline bug → drop_duplicates()
└── No  → key-level duplicates → inspect why
         ├── One should be the "truth" (latest update)  → sort + keep='first'
         ├── All rows have valid data (transactions)     → aggregate (sum, count, max)
         └── Caused by a join                            → fix the join, not the output
```

**Common mistakes:**
- Dropping duplicates before understanding why they exist — if caused by a bad join, the fix is upstream, not a `drop_duplicates` bandage
- Using `keep='first'` without sorting first — "first" means first in current row order, which may be arbitrary
- Checking only full duplicates and missing key-level duplicates — the more dangerous type is usually the key-level one

---
## §5 — Fix String & Categorical Inconsistencies

The same real-world entity appearing under different labels is one of the most common data quality issues. Groupby and merge operations silently fail when `"US"` and `"us"` are treated as different groups.

In [ ]:
# ── Step 1: Always inspect the raw unique values first ──────────────────────
df['country'].value_counts(dropna=False)
# Look for: trailing spaces, mixed case, abbreviations vs full names, typos

# ── Fix 1: Normalize case and whitespace — do this first, always ────────────
df['country'] = df['country'].str.strip().str.lower()
# Before: 'US', ' us', 'US ', 'us' → After: 'us' (all unified)

# ── Fix 2: Map known variants to a canonical value ──────────────────────────
country_map = {
    'us':            'United States',
    'usa':           'United States',
    'united states': 'United States',
    'u.s.':          'United States',
    'uk':            'United Kingdom',
    'united kingdom':'United Kingdom',
    'gb':            'United Kingdom',
}
df['country'] = df['country'].map(country_map).fillna(df['country'])
# .fillna(df['country']) preserves values not in the map (don't silently nullify them)

# ── Fix 3: Replace known typos ──────────────────────────────────────────────
typo_map = {
    'cnaada':  'canada',
    'germny':  'germany',
    'brazill': 'brazil',
}
df['country'] = df['country'].replace(typo_map)

# ── Fix 4: Bucket sparse categories into 'Other' ───────────────────────────
# Use when: long tail of rare categories that don't merit their own group
top_countries = df['country'].value_counts().nlargest(10).index
df['country_grouped'] = df['country'].where(
    df['country'].isin(top_countries), other='Other'
)

# ── Fix 5: Standardize boolean-like categoricals ─────────────────────────────
# Common in survey or CRM data
bool_map = {
    'yes': 1, 'no': 0,
    'y':   1, 'n':  0,
    '1':   1, '0':  0,
    'true':1, 'false': 0
}
df['subscribed'] = df['subscribed'].str.strip().str.lower().map(bool_map)

# ── Fix 6: Enforce a controlled vocabulary ──────────────────────────────────
# Catch unexpected values after cleaning
valid_platforms = {'ios', 'android', 'web', 'other'}
unexpected = set(df['platform'].unique()) - valid_platforms
if unexpected:
    print(f'WARNING: unexpected platform values: {unexpected}')

**Cleaning sequence — always in this order:**

1. `.str.strip()` — remove whitespace
2. `.str.lower()` — normalize case
3. `.replace(typo_map)` — fix known typos
4. `.map(canonical_map)` — unify synonyms
5. Bucket rare values into `'Other'`
6. Validate against controlled vocabulary

**Common mistakes:**
- Using `.map()` without `.fillna(df['col'])` — values not in the map become NaN, silently nullifying valid data
- Normalizing case after mapping — map first on lowercase, not on mixed-case raw values
- Forgetting that `pd.Series.replace()` replaces exact matches only; `.str.replace()` uses substring/regex matching — they are different methods

---
## §6 — Handle Outliers

Outliers are not always errors. The decision is: is this value impossible, implausible, or just extreme? Each case requires a different response.

In [ ]:
# ── Step 1: Detect outliers ──────────────────────────────────────────────────

# Method A: IQR — robust, distribution-free
Q1  = df['revenue'].quantile(0.25)
Q3  = df['revenue'].quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR
outliers_iqr = df[(df['revenue'] < lower) | (df['revenue'] > upper)]
print(f'IQR outliers: {len(outliers_iqr)} rows')

# Method B: Z-score — works well for roughly normal distributions
# Compute index-aligned: passing .dropna() into df[...] makes the mask shorter than
# df and raises "Item wrong length". Done this way, NaN rows become False in the mask.
z = (df['revenue'] - df['revenue'].mean()) / df['revenue'].std()
outliers_z = df[z.abs() > 3]
print(f'Z-score outliers (|z|>3): {len(outliers_z)} rows')

# Method C: Domain threshold — clearest when business rules exist
outliers_domain = df[(df['age'] < 0) | (df['age'] > 120)]

# ── Step 2: Decide what to do ───────────────────────────────────────────────

# Treatment A: Remove — only if value is impossible or a clear data error
df = df[(df['age'] >= 0) & (df['age'] <= 120)]

# Treatment B: Winsorize (cap) — clip to plausible bounds, keep the row
# Use when: extreme but real values exist (e.g. one user spent $1M)
p01 = df['revenue'].quantile(0.01)
p99 = df['revenue'].quantile(0.99)
df['revenue_capped'] = df['revenue'].clip(lower=p01, upper=p99)

# Treatment C: Flag and keep — preserve the outlier, mark it for downstream
# Use when: the outlier may be real and important (e.g. a whale customer)
df['is_revenue_outlier'] = (df['revenue'] > upper).astype(int)

# Treatment D: Log transform — compress the scale, don't remove
# Use when: data is right-skewed (common for spend, income, counts)
df['log_revenue'] = np.log1p(df['revenue'])   # log1p = log(x+1), handles zeros safely

# ── Summary: check the effect of your treatment ─────────────────────────────
print(df['revenue'].describe())
print(df['revenue_capped'].describe())

**Treatment decision guide:**

| Outlier type | Treatment | Reason |
| :--- | :--- | :--- |
| Impossible value (age = -5) | Remove | Data entry error — the row is invalid |
| Plausible but extreme (spend = $1M) | Winsorize or flag | Real event — removing distorts the picture |
| Right-skewed distribution | Log transform | Compresses scale without losing data |
| Extreme but informative (whale user) | Flag + keep | May be the most important segment |

**Common mistakes:**
- Removing outliers reflexively — always ask "is this impossible or just unexpected?"
- Using Z-score on heavily skewed data — Z-score assumes normality; IQR is more robust for skewed distributions
- Winsorizing before splitting train/test — compute the percentile bounds on train set only, then apply to test to avoid leakage
- Using `np.log()` on columns with zeros — raises a warning and produces `-inf`; always use `np.log1p()`

---
## §7 — Clean Column Names

Messy column names break `.query()`, attribute-style access (`df.column_name`), and SQL integrations. Fix them once, right after loading.

In [ ]:
import re

# ── The standard cleaning pipeline ──────────────────────────────────────────
def clean_column_names(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = (
        df.columns
          .str.strip()                        # remove leading/trailing whitespace
          .str.lower()                        # lowercase everything
          .str.replace(r'[^\w\s]', '', regex=True)  # remove special chars (keep letters, digits, _)
          .str.replace(r'\s+', '_', regex=True)      # spaces → underscores
          .str.replace(r'_+', '_', regex=True)       # collapse multiple underscores
          .str.strip('_')                     # remove leading/trailing underscores
    )
    return df

df = clean_column_names(df)

# Example transformation:
# ' Total Revenue ($) ' → 'total_revenue'   (trailing _ removed by .str.strip('_'))
# 'User ID'             → 'user_id'
# 'Q1 2024 Sales'       → 'q1_2024_sales'
# '__internal__'        → 'internal'

# ── Handle duplicate column names after cleaning ─────────────────────────────
# Can happen when 'Revenue $' and 'Revenue %' both become 'revenue'
cols = pd.Series(df.columns)
for dup in cols[cols.duplicated()].unique():
    dup_idx = cols[cols == dup].index.tolist()
    for i, idx in enumerate(dup_idx):
        cols[idx] = f'{dup}_{i}'
df.columns = cols

# ── Rename specific columns with a map ──────────────────────────────────────
df = df.rename(columns={
    'cust_id': 'customer_id',
    'rev':     'revenue',
    'dt':      'event_date'
})

# ── Validate final column names ──────────────────────────────────────────────
bad_cols = [c for c in df.columns if not re.match(r'^[a-z][a-z0-9_]*$', c)]
if bad_cols:
    print(f'WARNING: non-standard column names remain: {bad_cols}')

**Common mistakes:**
- Cleaning column names in the middle of a pipeline — do it immediately after loading, before any selection or filtering
- Forgetting to handle duplicate column names after cleaning — two different columns silently collapse to the same name
- Using `.str.replace(' ', '_')` without collapsing multiple spaces first — `'Total  Revenue'` becomes `'Total__Revenue'`

---
## Decision Guide

```
Just loaded a new dataset?
└── Always run: shape → .info() → null summary → duplicates → .describe() → value ranges  (§1)

Column dtype is wrong?
├── Date stored as string           → pd.to_datetime()                               (§2)
├── ID stored as float              → fill nulls → .astype(int) or .astype(str)      (§2)
├── Number has $, commas            → .str.replace() → .astype(float)                (§2)
└── String with <50 unique values   → .astype('category')                            (§2)

Column has missing values?
├── Null rate < 5%, random          → dropna(subset=[...])                           (§3)
├── Null means zero by logic        → fillna(0)                                      (§3)
├── Null should carry forward       → sort first → groupby().ffill()                 (§3)
├── Null depends on group           → groupby().transform(lambda x: x.fillna(...))   (§3)
├── Null is informative             → flag column + fill                             (§3)
└── Null rate > 60%                 → drop the column                                (§3)

Duplicate rows detected?
├── All columns identical           → drop_duplicates()                              (§4)
├── Same key, need latest record    → sort by date → drop_duplicates(keep='first')   (§4)
├── Same key, all rows are valid    → groupby().agg()                                (§4)
└── Caused by a join                → fix the join upstream                          (§4)

Category labels are inconsistent?
└── strip → lower → replace typos → map synonyms → bucket rare values               (§5)

Outliers detected?
├── Impossible value                → remove the row                                 (§6)
├── Plausible but extreme           → .clip() to winsorize                           (§6)
├── Right-skewed distribution       → np.log1p() transform                           (§6)
└── Extreme but meaningful          → flag column and keep                           (§6)

Column names are messy?
└── strip → lower → special chars → spaces→underscore → deduplicate                 (§7)
```